# 🏠 House Price Prediction using Regression Algorithms
**Project by: [Your Name] | Mind Bend Technologies**

---
### 📌 Project Overview
- Dataset: California Housing Dataset (from sklearn)
- Algorithms: Linear Regression, Ridge, Lasso, Random Forest Regressor
- Deployment: Hugging Face Spaces (Gradio)
---

## Step 1: Install Required Libraries

In [ ]:
# Install all required libraries
!pip install pandas numpy scikit-learn matplotlib seaborn joblib gradio

## Step 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

print('✅ All libraries imported successfully!')

## Step 3: Load Dataset

In [ ]:
# Load California Housing Dataset
housing = fetch_california_housing()

# Convert to DataFrame
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['Price'] = housing.target  # Target: House price in $100,000 units

print('Dataset Shape:', df.shape)
print('\nFeature Names:', housing.feature_names)
df.head()

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Basic Statistics ===')
df.describe()

In [ ]:
# Price Distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(df['Price'], bins=50, color='steelblue', edgecolor='black')
plt.title('House Price Distribution')
plt.xlabel('Price (in $100k)')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
sns.boxplot(y=df['Price'], color='coral')
plt.title('House Price Boxplot')
plt.ylabel('Price (in $100k)')

plt.tight_layout()
plt.savefig('price_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Plot saved!')

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 8))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Feature vs Price scatter plots
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
features = housing.feature_names

for i, feature in enumerate(features):
    row = i // 4
    col = i % 4
    axes[row, col].scatter(df[feature], df['Price'], alpha=0.3, s=5, color='steelblue')
    axes[row, col].set_xlabel(feature)
    axes[row, col].set_ylabel('Price')
    axes[row, col].set_title(f'{feature} vs Price')

plt.tight_layout()
plt.savefig('feature_vs_price.png', dpi=100, bbox_inches='tight')
plt.show()

## Step 5: Data Preprocessing

In [ ]:
# Features & Target
X = df.drop('Price', axis=1)
y = df['Price']

# Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set size: {X_train.shape}')
print(f'Test set size: {X_test.shape}')

# Feature Scaling (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('\n✅ Data preprocessing complete!')
print('Feature names:', list(X.columns))

## Step 6: Train Multiple Regression Models

In [ ]:
# Define all models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=0.01),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

# Train and evaluate each model
results = {}

for name, model in models.items():
    print(f'\n🔄 Training {name}...')
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {
        'MSE': round(mse, 4),
        'RMSE': round(rmse, 4),
        'MAE': round(mae, 4),
        'R2 Score': round(r2, 4)
    }
    print(f'   ✅ R2 Score: {r2:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}')

print('\n🎉 All models trained!')

## Step 7: Compare Model Performance

In [ ]:
# Results DataFrame
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('R2 Score', ascending=False)
print('=== Model Comparison ===')
print(results_df)
print(f'\n🏆 Best Model: {results_df.index[0]} (R2 = {results_df["R2 Score"].iloc[0]})')

In [ ]:
# Visualization: Model Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R2 Score comparison
colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(results_df))]
bars = axes[0].barh(results_df.index, results_df['R2 Score'], color=colors, edgecolor='black')
axes[0].set_xlabel('R2 Score')
axes[0].set_title('R2 Score Comparison (Higher is Better)')
axes[0].set_xlim(0, 1)
for bar, val in zip(bars, results_df['R2 Score']):
    axes[0].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=10)

# RMSE comparison
bars2 = axes[1].barh(results_df.index, results_df['RMSE'], color='coral', edgecolor='black')
axes[1].set_xlabel('RMSE')
axes[1].set_title('RMSE Comparison (Lower is Better)')
for bar, val in zip(bars2, results_df['RMSE']):
    axes[1].text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Step 8: Best Model - Detailed Analysis

In [ ]:
# Use best model (Random Forest usually wins)
best_model_name = results_df.index[0]
best_model = models[best_model_name]
y_pred_best = best_model.predict(X_test_scaled)

print(f'🏆 Best Model: {best_model_name}')

# Actual vs Predicted
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred_best, alpha=0.5, color='steelblue', s=15)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title(f'{best_model_name}\nActual vs Predicted')

plt.subplot(1, 2, 2)
residuals = y_test - y_pred_best
plt.hist(residuals, bins=50, color='coral', edgecolor='black')
plt.axvline(x=0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Residuals')
plt.ylabel('Frequency')
plt.title('Residual Distribution')

plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance (for Random Forest)
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feature_names = X.columns
    
    feat_imp_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=True)
    
    plt.figure(figsize=(8, 5))
    plt.barh(feat_imp_df['Feature'], feat_imp_df['Importance'], color='steelblue', edgecolor='black')
    plt.xlabel('Importance Score')
    plt.title('Feature Importance')
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=100, bbox_inches='tight')
    plt.show()
    print('\n=== Feature Importance ===')
    print(feat_imp_df.sort_values('Importance', ascending=False).to_string(index=False))

## Step 9: Save Model & Scaler

In [ ]:
# Save best model and scaler using joblib
joblib.dump(best_model, 'house_price_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print('✅ Model saved as: house_price_model.pkl')
print('✅ Scaler saved as: scaler.pkl')

# Verify by loading and predicting
loaded_model = joblib.load('house_price_model.pkl')
loaded_scaler = joblib.load('scaler.pkl')

# Test prediction
sample = X_test.iloc[:1]
sample_scaled = loaded_scaler.transform(sample)
test_pred = loaded_model.predict(sample_scaled)
print(f'\n🧪 Test Prediction: ${test_pred[0]*100000:.2f}')
print(f'🎯 Actual Value: ${y_test.iloc[0]*100000:.2f}')

## Step 10: Test Gradio App Locally (before HF deployment)

In [ ]:
import gradio as gr
import joblib
import numpy as np

# Load model and scaler
model = joblib.load('house_price_model.pkl')
scaler = joblib.load('scaler.pkl')

def predict_price(MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude):
    features = np.array([[MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude]])
    features_scaled = scaler.transform(features)
    prediction = model.predict(features_scaled)[0]
    return f"🏠 Predicted House Price: **${prediction * 100000:,.2f}**"

# Create Gradio Interface
demo = gr.Interface(
    fn=predict_price,
    inputs=[
        gr.Slider(0.5, 15.0, value=3.5, label="Median Income (in $10k)", step=0.1),
        gr.Slider(1, 52, value=20, label="House Age (years)", step=1),
        gr.Slider(1.0, 15.0, value=5.0, label="Average Rooms", step=0.1),
        gr.Slider(0.5, 5.0, value=1.1, label="Average Bedrooms", step=0.1),
        gr.Slider(100, 35000, value=1200, label="Population", step=100),
        gr.Slider(1.0, 10.0, value=3.0, label="Average Occupants", step=0.1),
        gr.Slider(32.0, 42.0, value=37.0, label="Latitude", step=0.1),
        gr.Slider(-124.0, -114.0, value=-120.0, label="Longitude", step=0.1),
    ],
    outputs=gr.Markdown(),
    title="🏠 House Price Predictor",
    description="Adjust the sliders to predict California house prices.",
    theme=gr.themes.Soft()
)

demo.launch(share=True)  # share=True gives public URL in Colab

## Step 11: Deploy to Hugging Face Spaces

In [ ]:
# Install huggingface_hub
!pip install huggingface_hub

In [ ]:
from huggingface_hub import HfApi, login

# Step 1: Login to Hugging Face
# Go to: https://huggingface.co/settings/tokens
# Create a token with 'write' permission
login()  # Will prompt for your HF token

In [ ]:
from huggingface_hub import HfApi
import os

# ⚠️ CHANGE THIS to your Hugging Face username
HF_USERNAME = "your-hf-username"  # <-- Change this!
SPACE_NAME = "house-price-predictor"
REPO_ID = f"{HF_USERNAME}/{SPACE_NAME}"

api = HfApi()

# Create the Space on Hugging Face
api.create_repo(
    repo_id=REPO_ID,
    repo_type="space",
    space_sdk="gradio",
    exist_ok=True
)
print(f'✅ Space created: https://huggingface.co/spaces/{REPO_ID}')

In [ ]:
# Upload all required files to the Space
files_to_upload = [
    'app.py',
    'requirements.txt',
    'house_price_model.pkl',
    'scaler.pkl'
]

for fname in files_to_upload:
    if os.path.exists(fname):
        api.upload_file(
            path_or_fileobj=fname,
            path_in_repo=fname,
            repo_id=REPO_ID,
            repo_type="space"
        )
        print(f'✅ Uploaded: {fname}')
    else:
        print(f'❌ File not found: {fname}')

print(f'\n🚀 Deployment Complete!')
print(f'🌐 Visit: https://huggingface.co/spaces/{REPO_ID}')